# Insulation and boundaries

Computes the insulation table (10 kb resolution, 50 kb and 100 kb windows) for
every cooler, caches it in `../data/insulation`, and writes one summary table
comparing each subsample with its own full-depth library at every boundary
tolerance.

In [1]:
import sys
import warnings

warnings.filterwarnings("ignore")

sys.path.insert(0, "other")
from hic_common import *

N_WORKERS = 12    # coolers computed at the same time
NPROC = 2        # threads inside 
#datasets and other settings in other/hic_common.py
print(f"windows: {INSUL_WINDOWS}")
print(f"tolerances: {TOLERANCES} bins ({RES_INSULATION} bp)")

windows: [50000, 100000]
tolerances: [1, 2, 3, 5] bins (10000 bp)


## Compute and cache insulation tables

In [2]:
tasks = build_tasks(NPROC)
print(f"{len(tasks)} coolers in total")

results = run_parallel(tasks, worker_insulation, N_WORKERS, "insul")

990 coolers in total
[1/990] insul skip         SRR13601502_full
[2/990] insul skip         SRR13601502_10000000_sample1
[3/990] insul skip         SRR13601502_100000000_sample3
[4/990] insul skip         SRR13601502_100000000_sample2
[5/990] insul skip         SRR13601502_10000000_sample10
[6/990] insul skip         SRR13601502_10000000_sample11
[7/990] insul skip         SRR13601502_100000000_sample4
[8/990] insul skip         SRR13601502_100000000_sample5
[9/990] insul skip         SRR13601502_100000000_sample1
[10/990] insul skip         SRR13601502_10000000_sample12
[11/990] insul skip         SRR13601502_10000000_sample14
[12/990] insul skip         SRR13601502_10000000_sample13
[13/990] insul skip         SRR13601502_10000000_sample15
[14/990] insul skip         SRR13601502_10000000_sample16
[15/990] insul skip         SRR13601502_10000000_sample17
[16/990] insul skip         SRR13601502_10000000_sample18
[17/990] insul skip         SRR13601502_10000000_sample19
[18/990] insul s

## Compare every subsample with its full-depth library

In [3]:
import multiprocessing

SUMMARY_WORKERS = 24        

_reference_cache = {}


def boundaries(insulation, window):
    '''Positions of the called boundaries for one window.'''
    return insulation.loc[insulation[f"is_boundary_{window}"] == True,
                          ["chrom", "start", "end"]]


def mean_called_strength(insulation, window):
    '''Mean strength over called boundaries only; NaN if none were called.'''
    called = insulation[f"is_boundary_{window}"] == True
    return (float(insulation.loc[called, f"boundary_strength_{window}"].mean())
            if called.any() else np.nan)


def reference_for(sample_name, srr):
    '''Full-depth insulation of one library, kept in the worker between tasks.'''
    key = (sample_name, srr)
    if key not in _reference_cache:
        _reference_cache.clear()                
        _reference_cache[key] = load_insulation(sample_name, f"{srr}_full")
    return _reference_cache[key]


def summarize_entity(task):
    '''All summary rows (windows x tolerances) for one cooler.'''
    sample_name, srr, entity = task
    try:
        reference = reference_for(sample_name, srr)
        insulation = load_insulation(sample_name, entity["entity_id"])
        rows = []
        for window in INSUL_WINDOWS:
            correlation = track_correlations(
                insulation, reference, f"log2_insulation_score_{window}")
            test_boundaries = boundaries(insulation, window)
            reference_boundaries = boundaries(reference, window)
            strength = mean_called_strength(insulation, window)

            for tolerance in TOLERANCES:
                match = match_boundary_sets(test_boundaries, reference_boundaries,
                                            RES_INSULATION, tolerance)
                rows.append(dict(sample_name=sample_name, srr=srr,
                                 entity_id=entity["entity_id"], role=entity["role"],
                                 depth=entity["depth"], sample_idx=entity["sample_idx"],
                                 window=window, tol_bins=tolerance,
                                 ins_pearson=correlation["pearson"],
                                 ins_spearman=correlation["spearman"],
                                 n_bins=correlation["n_bins"],
                                 n_boundaries=len(test_boundaries),
                                 mean_boundary_strength=strength,
                                 b_precision=match["precision"],
                                 b_recall=match["recall"],
                                 b_f1=match["f1"]))
        return rows
    except Exception as error:
        print(f"FAILED {entity['entity_id']}: {error!r}", flush=True)
        return []


tasks = []
for sample_name, srr in LIBRARIES:
    if not is_cached(insulation_path(sample_name, f"{srr}_full")):
        print(f"no full-depth insulation for {sample_name} / {srr}")
        continue
    for entity in list_entities(sample_name, srr):
        if is_cached(insulation_path(sample_name, entity["entity_id"])):
            tasks.append((sample_name, srr, entity))

print(f"{len(tasks)} coolers to summarize")

rows = []
context = multiprocessing.get_context("fork")
with context.Pool(processes=SUMMARY_WORKERS) as pool:
    for n, result in enumerate(pool.imap(summarize_entity, tasks, chunksize=8), 1):
        rows.extend(result)
        if n % 50 == 0 or n == len(tasks):
            print(f"[{n}/{len(tasks)}]", flush=True)

summary = pd.DataFrame(rows)
save_parquet(summary, DATASETS_DIR / "insulation_summary.parquet")

expected = len(tasks) * len(INSUL_WINDOWS) * len(TOLERANCES)
print(f"rows: {len(summary)} (expected {expected})")
display(summary.head())

765 coolers to summarize
[50/765]
[100/765]
[150/765]
[200/765]
[250/765]
[300/765]
[350/765]
[400/765]
[450/765]
[500/765]
[550/765]
[600/765]
[650/765]
[700/765]
[750/765]
[765/765]
rows: 6120 (expected 6120)


,sample_name,srr,entity_id,role,depth,sample_idx,window,tol_bins,ins_pearson,ins_spearman,n_bins,n_boundaries,mean_boundary_strength,b_precision,b_recall,b_f1
0,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,50000,1,1.0,1.0,263486,14472,0.725447,1.0,1.0,1.0
1,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,50000,2,1.0,1.0,263486,14472,0.725447,1.0,1.0,1.0
2,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,50000,3,1.0,1.0,263486,14472,0.725447,1.0,1.0,1.0
3,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,50000,5,1.0,1.0,263486,14472,0.725447,1.0,1.0,1.0
4,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,100000,1,1.0,1.0,265214,9395,0.815313,1.0,1.0,1.0


In [4]:
display(summary.tail())

,sample_name,srr,entity_id,role,depth,sample_idx,window,tol_bins,ins_pearson,ins_spearman,n_bins,n_boundaries,mean_boundary_strength,b_precision,b_recall,b_f1
6115,kbm7,SRR1658708,SRR1658708_50000000_sample9,subsample,50000000,9,50000,5,0.583830,0.596956,266185,19821,1.134486,0.752283,0.818650,0.784065
6116,kbm7,SRR1658708,SRR1658708_50000000_sample9,subsample,50000000,9,100000,1,0.778879,0.766208,268085,13247,0.796237,0.374047,0.521314,0.435569
6117,kbm7,SRR1658708,SRR1658708_50000000_sample9,subsample,50000000,9,100000,2,0.778879,0.766208,268085,13247,0.796237,0.479278,0.666667,0.557651
6118,kbm7,SRR1658708,SRR1658708_50000000_sample9,subsample,50000000,9,100000,3,0.778879,0.766208,268085,13247,0.796237,0.543972,0.750763,0.630854
6119,kbm7,SRR1658708,SRR1658708_50000000_sample9,subsample,50000000,9,100000,5,0.778879,0.766208,268085,13247,0.796237,0.631464,0.850753,0.724887
